# Kaiku Finance - pipeline test (20-call budget)

Every question follows the same fixed path, with **at most 2 model calls**:

1. **Route** - one small Gemini call returns a typed decision. A code guardrail can only move it towards the web.
2. **Retrieve** - code only: Exa search and/or pandas totals.
3. **Answer** - one Gemini call that sees only the evidence. Skipped for `static` (the router already answered from training).

| Section | Model calls |
|---|---|
| 1. Guardrail check (offline) | 0 |
| 2. Routing of 10 questions (router only) | 10 |
| 3. Full answers for 4 of them (routing is cached) | 4 |
| 4. A question with no real answer | 2 |
| 5. Exa settings | 0 |
| 6. Your own question | up to 2 |
| **Total** | **18** |

Each section checks the budget before calling the model. Needs `GEMINI_API_KEY` and `EXA_API_KEY` in `.env`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # notebook/ -> ../src

import pandas as pd
from project_aethon.kaiku_agent import (MODEL_NAME, create_agent, format_evidence, load_transactions,
                                        router_prompt, search_exa, ANSWER_PROMPT)
from project_aethon.kaiku_eval import (QUESTIONS, UNANSWERABLE, ask_all, compare_exa_configs,
                                       guardrail_table, remaining, results_table, routing_eval, show)

pd.set_option("display.max_colwidth", 120)

MODEL = MODEL_NAME     # the daily quota is per model: change it here if this one is exhausted
BUDGET = 20            # model calls allowed for the whole notebook

transactions = load_transactions()   # synthetic sample; pass a CSV path for the S-Bank dataset
agent = create_agent(model_name=MODEL, transactions=transactions)
print(f"Model: {MODEL} | prompt sizes (chars/4): router ~{len(router_prompt(agent.categories)) // 4} tokens, "
      f"answer ~{len(ANSWER_PROMPT) // 4} tokens | calls used: {agent.stats['llm_calls']}/{BUDGET}")

Model: gemini-3.5-flash | prompt sizes (chars/4): router ~327 tokens, answer ~130 tokens | calls used: 0/20


## 1. Guardrail check (0 calls)

The code rule that sends a question to Exa no matter what the router says. Questions expected to be
`static` must show an empty `forced_to_web`.

In [2]:
guardrail_table()

,question,expected,forced_to_web,ok
0,What is a variable interest rate?,static,,True
1,How does compound interest work?,static,,True
2,Which national team won the 2022 World Cup?,static,,True
3,What is the 12-month Euribor today?,web,time marker: today,True
4,How much is the Kela general housing allowance this year for a single,web,time marker: this year,True
5,Can my landlord in Finland raise my rent by 8% in the middle of the co,web,place or institution: finland,True
6,"Write a short letter to my landlord objecting to a rent increase, citi",web,place or institution: finland,True
7,How much did I spend on taxis last month?,data,,True
8,Where is my money going? Split it into the three baskets.,data,,True
9,"How much do I pay in rent each month, and what is the average rent in",both,time marker: right now,True


## 2. Routing of 10 questions (10 calls)

Only the router runs: one call per question. Decisions are cached for section 3.

In [3]:
routing = routing_eval(agent, QUESTIONS, budget=BUDGET)
done = routing[routing["route"] != "NOT RUN"]
print(f"\nrouted: {len(done)}/{len(routing)} | correct: {done['ok'].mean():.2f} | "
      f"mean tokens: {done['tokens'].mean():.0f} | mean seconds: {done['seconds'].mean():.2f} | "
      f"calls used: {agent.stats['llm_calls']}/{BUDGET}")
routing

[OK] static  <- What is a variable interest rate?
[OK] static  <- How does compound interest work?
[OK] static  <- Which national team won the 2022 World Cup?
[OK] web     <- What is the 12-month Euribor today?
[OK] web     <- How much is the Kela general housing allowance this year for a single 
[OK] web     <- Can my landlord in Finland raise my rent by 8% in the middle of the co
[OK] web     <- Write a short letter to my landlord objecting to a rent increase, citi
[OK] data    <- How much did I spend on taxis last month?
[OK] data    <- Where is my money going? Split it into the three baskets.
[OK] both    <- How much do I pay in rent each month, and what is the average rent in 

routed: 10/10 | correct: 1.00 | mean tokens: 485 | mean seconds: 7.35 | calls used: 10/20


,question,expected,route,ok,decided_by,query,official,tokens,seconds
0,What is a variable interest rate?,static,static,True,model,NaN,None,512,2.63
1,How does compound interest work?,static,static,True,model,NaN,None,471,3.50
2,Which national team won the 2022 World Cup?,static,static,True,model,NaN,None,501,1.94
3,What is the 12-month Euribor today?,web,web,True,model,12 month Euribor rate November 2026,True,476,4.98
4,How much is the Kela general housing allowance this year for,web,web,True,model,Kela general housing allowance single person Helsinki 2026,True,482,1.70
5,Can my landlord in Finland raise my rent by 8% in the middle,web,web,True,model,Finland rent increase rules lease contract legal limit,True,478,1.75
6,Write a short letter to my landlord objecting to a rent incr,web,web,True,model,Finland tenant rights legal rules for rent increase objection letter,True,482,6.53
7,How much did I spend on taxis last month?,data,data,True,model,NaN,None,483,6.63
8,Where is my money going? Split it into the three baskets.,data,data,True,model,NaN,None,464,40.42
9,"How much do I pay in rent each month, and what is the averag",both,both,True,model,average rent per square meter Helsinki Finland 2026,False,505,3.45


## 3. Full answers (4 calls)

One case per route. The routing comes from the cache, so each case costs one answer call;
the `static` case costs nothing more because the router already wrote its answer.

In [4]:
CASES = {
    "static": QUESTIONS[0][0],
    "web": QUESTIONS[3][0],
    "letter": QUESTIONS[6][0],
    "data": QUESTIONS[7][0],
    "both": QUESTIONS[9][0],
}
results = ask_all(agent, CASES, budget=BUDGET)
print(f"calls used: {agent.stats['llm_calls']}/{BUDGET}")
results_table(results)

calls used: 14/20


,case,route,decided_by,calls,tokens,router_tokens,answer_tokens,thinking,sources,cited,exa_usd,seconds,label_problems
0,static,static,model,1,512,512,0,0,0,0,0.000,2.63,
1,web,web,model,2,1089,476,613,123,3,2,0.007,12.39,
2,letter,web,model,2,1594,482,1112,606,3,3,0.007,14.71,
3,data,data,model,2,840,483,357,111,0,0,0.000,9.80,
4,both,both,model,2,1802,505,1297,536,3,1,0.007,12.00,


In [5]:
for name, res in results.items():
    print("=" * 30, name, "=" * 30)
    show(res)
    print()

============================== static ==============================
route: static | decided by: model | calls: 1 | tokens: 512 (router 374+138, answer 0+0, thinking 0) | 2.63s {'route': 2.63}

A variable interest rate is an interest rate on a loan or investment that can change over time. It fluctuates based on an underlying benchmark rate or index, such as the Euribor. This means your periodic interest payments can go up or down over the term of the financial product. [MODEL]

============================== web ==============================
route: web | decided by: model | calls: 2 | tokens: 1089 (router 379+97, answer 431+182, thinking 123) | 12.39s {'route': 4.98, 'retrieve': 3.99, 'answer': 3.42}
search: '12 month Euribor rate November 2026' | official=True | 3 sources | Exa $0.007

The 12-month Euribor rate is 3.328% as of 13 November 2026 [WEB 2]. Euribor rates measure the interest rate at which European banks obtain euro-denominated money market financing without collateral [WE

## 4. A question with no real answer (2 calls)

The answer should say that the evidence does not cover it, with no invented figure.

In [6]:
if remaining(agent, BUDGET) >= 2:
    show(agent.ask(UNANSWERABLE))
    print(f"\ncalls used: {agent.stats['llm_calls']}/{BUDGET}")
else:
    print(f"Skipped: {remaining(agent, BUDGET)} calls left.")

route: web | decided by: model | calls: 2 | tokens: 973 (router 392+61, answer 402+118, thinking 81) | 9.64s {'route': 4.15, 'retrieve': 1.62, 'answer': 3.87}
search: 'Finland Kela benefits Zzyzx municipality blue hamsters 2031' | official=True | 3 sources | Exa $0.007

I am sorry, but the provided evidence does not contain information about Kela benefits for owners of blue hamsters or specific amounts for Zzyzx municipality in 2031.

LABEL PROBLEMS: ['web evidence but no [WEB n] label']

calls used: 16/20


## 5. Exa settings (0 model calls)

These cells spend Exa credits only, so they can be re-run. `evidence_tokens` is what one search adds
to the answer call. Check the expected texts in `EXA_EVAL` before trusting `hit_rate`.

In [7]:
raw = search_exa("Kela general housing allowance amount", num_results=3, highlight_chars=300)
print("Exa cost (USD):", raw["cost"])
print(format_evidence(raw["results"], 300))

Exa cost (USD): 0.007
[1] kela.fi | How do income and assets affect general housing ... | 2010-02-01
"## How is housing allowance calculated? General housing allowance covers a maximum of 70% of your acceptable housing costs. The amount of housing allowance is calculated as follows: 0.7 x (acceptable housing costs – basic deductible). Because there are several factors that affect the housing allowan"
[2] laskurit.kela.fi | General housing allowance calculator
"## How to use the calculator With this calculator, you can estimate the amount of general housing allowance. Information marked with an asterisk (*) is required. ... as two persons for the purpose of determining the upper limit for the housing costs"
[3] infofinland.fi | Housing allowance
"The general housing allowance is 70% of the eligible housing costs, after deducting the basic deductible. If you spend less money on housing than the eligible housing costs, the allowance amount is calculated on the basis of actual expenditure.

In [8]:
compare_exa_configs()

,config,hit_rate,evidence_tokens,exa_cost_usd
0,highlights 300 x3 (default),1.00,238.17,0.04
1,highlights 150 x3,0.83,148.83,0.04
2,highlights 300 x2,0.67,162.67,0.04
3,highlights 500 x3,1.00,320.00,0.04
4,full text 3000 x3,0.83,1958.50,0.04
5,highlights 300 x3 official,0.83,233.83,0.04
6,highlights 300 x3 lite,0.83,237.50,0.02


## 6. Your own question (up to 2 calls)

Asking the same question twice is free: the second answer comes from the cache.

In [9]:
QUESTION = "What happens to my budget if I stop ordering food delivery?"

if remaining(agent, BUDGET) >= 2:
    show(agent.ask(QUESTION))
else:
    print(f"Skipped: {remaining(agent, BUDGET)} calls left.")
print(f"\ncalls used: {agent.stats['llm_calls']}/{BUDGET} | totals: {agent.stats}")

route: data | decided by: model | calls: 2 | tokens: 951 (router 379+52, answer 327+193, thinking 108) | 10.54s {'route': 4.59, 'retrieve': 0.01, 'answer': 5.94}
data: {"period": "2026-05-20 to 2026-11-07", "category": "food_delivery", "total_eur": 1242.07, "transactions": 45, "by_month": {"2026-05": 113.85, "2026-06": 212.33, "2026-07": 270.02, "2026-08": 195.06, "2026-09": 255.77, "2026-10": 163.02, "2026-11": 32.02}}

If you stop ordering food delivery, you will save a total of 1,242.07 EUR based on your recent spending history [DATA]. This total comes from 45 food delivery transactions made between May 20, 2026, and November 7, 2026 [DATA]. Eliminating this expense will keep those funds in your budget across upcoming months [DATA].

calls used: 18/20 | totals: {'llm_calls': 18, 'exa_calls': 4, 'input_tokens': 7026, 'output_tokens': 3131, 'exa_cost': 0.028}
